In [1]:
# Ensure jacopy is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# jacopy isn't already installed into this kernel.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

# 10, Diagnostics & proof debugging

Companion notebook to [10_diagnostics.md](10_diagnostics.md). What to do when `prove_equivalence` raises `ProofFailure`, read the residual, inspect the diagnostic report, extend the rule catalogue.

## A failing proof

Build an `ExpansionEngine` deliberately missing the d² = 0 rule. Forming `d(d(ω)) == 0` should close trivially in the default engine but stalls here, so we get a real failure to look at.

In [2]:
from jacopy.algebra.derivation import Act
from jacopy.calculus.exterior_d import d
from jacopy.core.expr import Symbol, Integer
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.proof import prove_equivalence, ProofFailure
from jacopy.proof.expansion import (
    ExpansionEngine,
    LieDerivativeCartanDefinition,
    ActOverSumOpDefinition,
    IotaSquaredZeroDefinition,
    IotaOnZeroFormDefinition,
    IotaOnExactOneFormDefinition,
)

reg = PropertyRegistry()
omega = Symbol("ω"); reg.declare(omega, Graded(degree=2))

engine_no_d2 = ExpansionEngine([
    LieDerivativeCartanDefinition(),
    ActOverSumOpDefinition(),
    IotaSquaredZeroDefinition(),
    IotaOnZeroFormDefinition(),
    IotaOnExactOneFormDefinition(d=d),
])

try:
    prove_equivalence(
        Act(d, Act(d, omega)),
        Integer(0),
        registry=reg,
        engine=engine_no_d2,
    )
except ProofFailure as exc:
    print(exc)

ExpandAndSimplify left residual d(d(ω)) when proving d(d(ω)) == 0

Residual: d(d(ω))
Hints (1):
  1. [stalled-d-squared] at d(d(ω)): d applied twice, an odd derivation should square to zero
     suggestion: enable d_squared_mode="axiom" on default_engine or register a DSquaredZeroDefinition for this operator


The exception message is two-tier, a one-line summary (`ExpandAndSimplify left residual ...`) followed by the structured `DiagnosticReport` block. The summary tells you *that* the proof stalled; the report tells you *why* the engine couldn't reduce the residual further.

## Reading the report

`ProofFailure.report` is a `DiagnosticReport` (or `None`). It carries `report.residual` (the surviving `Expr`) and a list of `DiagnosticHint`s, each with `category`, `message`, `location`, and `suggestion` fields.

In [3]:
try:
    prove_equivalence(
        Act(d, Act(d, omega)), Integer(0),
        registry=reg, engine=engine_no_d2,
    )
except ProofFailure as exc:
    report = exc.report
    print(f"residual : {report.residual}")
    print(f"hints    : {len(report)}")
    for hint in report:
        print(f"  [{hint.category}] {hint.message}")
        if hint.suggestion is not None:
            print(f"    fix: {hint.suggestion}")

residual : d(d(ω))
hints    : 1
  [stalled-d-squared] d applied twice, an odd derivation should square to zero
    fix: enable d_squared_mode="axiom" on default_engine or register a DSquaredZeroDefinition for this operator


## Calling `diagnose()` directly

You don't need a `ProofFailure` to use the diagnostic layer. `diagnose(expr, registry=...)` runs the full rule catalogue against any expression, useful for inspecting intermediate terms.

In [4]:
from jacopy.core.expr import Product
from jacopy.proof import diagnose

mystery = Symbol("M")  # never registered
report = diagnose(Product(mystery, omega), registry=reg)
print(report.format())

Residual: (M * ω)
Hints (1):
  1. [unclassified-factor] at M: factor M has no grading evidence (neither Scalar/Graded in registry nor a Derivation self-degree)
     suggestion: declare registry.declare(factor, Graded(degree=...)) or registry.declare(factor, Scalar())


## Built-in rule catalogue

Rules in `jacopy.proof.diagnostic_rules` register themselves on import. Each fires on a specific *stalled shape*:

| Category | Trigger |
|---|---|
| `stalled-d-squared` | `Act(d, Act(d, x))` survived |
| `stalled-iota-squared` | `Act(ι_X, Act(ι_X, x))` survived |
| `stalled-act-over-zero` | `Act(op, 0)` reached the residual |
| `stalled-act-over-neg-op` | `Act(Neg(op), x)` failed to peel |
| `unreduced-iota-on-df` | `ι_V(d(f))` with V a sum/product of derivations |
| `unclassified-factor` | a `Product` factor has no grading evidence |
| `symbol-vector-field` | a bare `Symbol` plays a vector-field role |

Filter to a single category with `report.by_category(...)`.

In [5]:
# Trigger several categories side by side, just by passing
# different residual shapes through diagnose().
from jacopy.core.expr import Sum, Neg

shapes = [
    ("d² stall",     Act(d, Act(d, omega))),
    ("Act on 0",     Act(d, Integer(0))),
    ("unclassified", Product(Symbol("M"), omega)),
]
for label, expr in shapes:
    rep = diagnose(expr, registry=reg)
    cats = sorted({h.category for h in rep})
    print(f"{label:14s} → categories: {cats}")

d² stall       → categories: ['stalled-d-squared']
Act on 0       → categories: ['stalled-act-over-zero']
unclassified   → categories: ['unclassified-factor']


## Adding your own rule

A rule is `(expr, registry, engine) → Iterable[DiagnosticHint]`. Register it with `@register_rule`, it joins the catalogue immediately, no engine wiring needed because diagnostics are read-only on the residual tree.

In [6]:
from jacopy.proof import DiagnosticHint, register_rule

@register_rule
def warn_on_unregistered_omega(expr, registry, engine):
    """Toy rule: flag bare 'ω' that lost its Graded declaration."""
    if registry is None:
        return
    stack = [expr]
    while stack:
        cur = stack.pop()
        if isinstance(cur, Symbol) and cur.name == "ω":
            if not registry.has(cur, Graded):
                yield DiagnosticHint(
                    category="omega-unregistered",
                    message="ω appears without a registered grading",
                    location=cur,
                    suggestion="reg.declare(ω, Graded(degree=...))",
                )
        stack.extend(getattr(cur, "children", ()))

# Exercise the new rule on an empty registry
empty = PropertyRegistry()
report = diagnose(omega, registry=empty)
for h in report:
    if h.category == "omega-unregistered":
        print(h)

DiagnosticHint(category='omega-unregistered', message='ω appears without a registered grading', location=ω, suggestion='reg.declare(ω, Graded(degree=...))')


## Summary

* `prove_equivalence` raises `ProofFailure`; `exc.report` is a `DiagnosticReport`.
* The report bundles `residual` + a list of `DiagnosticHint`s with `category` / `message` / `location` / `suggestion`.
* `diagnose(expr, registry=...)` runs the same pipeline directly on any `Expr`.
* Built-in catalogue covers d² / ι² stalls, `Act` linearity gaps, unreduced iota-on-df, and unclassified factors.
* Extend the catalogue with `@register_rule`, pure tree-walk function, no engine plumbing.